# Patch-Based Detection of AI-Generated Facial Images

**Authors:** Mark Gislason, Samia Jaman

This notebook investigates whether convolutional neural networks (CNNs) can distinguish AI-generated facial images from real photographs using **patch-based classification**.

Rather than relying only on an entire face, the patch models divide each 128×128 image into smaller regions and classify those regions independently. Patch predictions are then aggregated with majority voting to obtain an image-level prediction.

### Research questions

1. How does patch size affect classification accuracy and training time?
2. How do patch-based CNNs compare with a full-image CNN?
3. How does reducing the amount of training data affect each approach?
4. Can patch-level predictions provide interpretable spatial evidence for a classification?

The experiments compare 16×16, 32×32, 64×64, and 128×128 inputs, along with a fully connected 32×32 baseline.

## 1. Setup

The original experiments were run in Google Colab with an A100 GPU. The project uses PyTorch for modeling and `face_recognition` during preprocessing.

If `face_recognition` is not already installed in your environment, install it before running the notebook.

In [ ]:
# Uncomment if needed:
# %pip install face_recognition

from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Data

The original dataset contained 140,000 256×256 RGB facial images: 70,000 AI-generated StyleGAN faces and 70,000 real Flickr face images.

During preprocessing, faces were localized with `face_recognition`, problematic images were removed, and the remaining images were cropped to a centered 128×128 region. The experiments used 69,500 real and 69,500 fake images.

For GitHub portability, set `DATA_DIR` to a folder containing:

```text
data/
├── real/
│   ├── 000000.jpg
│   └── ...
└── fake/
    ├── 000000.jpg
    └── ...
```

The dataset itself is not stored in this repository.

In [ ]:
DATA_DIR = Path("data")

REAL_DIR = DATA_DIR / "real"
FAKE_DIR = DATA_DIR / "fake"

N_PER_CLASS = 69_500

paths = [REAL_DIR / f"{i:06d}.jpg" for i in range(N_PER_CLASS)]
labels = [0] * N_PER_CLASS

paths += [FAKE_DIR / f"{i:06d}.jpg" for i in range(N_PER_CLASS)]
labels += [1] * N_PER_CLASS

print(f"Configured {len(paths):,} image paths.")

## 3. Train, Validation, and Test Split

The data is split **before patch generation** so that patches from the same source image cannot leak across training, validation, and test sets.

The primary experiment uses an 80/10/10 split. Additional training subsets containing 25% and 10% of the original training set are created for the reduced-data experiments.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    paths,
    labels,
    test_size=0.20,
    stratify=labels,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

X_25, _, y_25, _ = train_test_split(
    X_train,
    y_train,
    test_size=0.75,
    stratify=y_train,
    random_state=42,
)

X_10, _, y_10, _ = train_test_split(
    X_train,
    y_train,
    test_size=0.90,
    stratify=y_train,
    random_state=42,
)

print(f"Train:      {len(X_train):,}")
print(f"Validation: {len(X_val):,}")
print(f"Test:       {len(X_test):,}")

## 4. Patch Dataset

`GridPatchDataset` dynamically extracts non-overlapping patches from each 128×128 image. This avoids writing every patch to disk and keeps patch generation inside the PyTorch data pipeline.

In [ ]:
class GridPatchDataset(Dataset):
    def __init__(self, image_paths, labels, patch_size, image_size=128):
        if image_size % patch_size != 0:
            raise ValueError("patch_size must evenly divide image_size.")

        self.paths = image_paths
        self.labels = labels
        self.patch_size = patch_size
        self.image_size = image_size
        self.grid_size = image_size // patch_size
        self.transform = transforms.ToTensor()

        self.index = [
            (img_idx, row, col)
            for img_idx in range(len(image_paths))
            for row in range(self.grid_size)
            for col in range(self.grid_size)
        ]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        img_idx, row, col = self.index[idx]
        ps = self.patch_size

        image = Image.open(self.paths[img_idx]).convert("RGB")

        left = col * ps
        upper = row * ps
        patch = image.crop((left, upper, left + ps, upper + ps))

        return self.transform(patch), self.labels[img_idx], img_idx

## 5. Data Loaders

Batch sizes differ by patch size so that the experiments can process a comparable number of patch samples per optimization step. The helper below replaces the repeated loader blocks from the working notebook while preserving the same experimental setup.

In [ ]:
BATCH_SIZES = {
    16: 1024,
    32: 256,
    64: 64,
    128: 16,
}


def make_loaders(
    patch_size,
    train_paths=X_train,
    train_labels=y_train,
    num_workers=8,
):
    batch_size = BATCH_SIZES[patch_size]

    train_dataset = GridPatchDataset(train_paths, train_labels, patch_size)
    val_dataset = GridPatchDataset(X_val, y_val, patch_size)
    test_dataset = GridPatchDataset(X_test, y_test, patch_size)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return train_loader, val_loader, test_loader

## 6. Model Architectures

### Fully Connected Baseline

The 32×32 baseline flattens each RGB patch and passes it through four fully connected layers. Because flattening discards explicit spatial relationships among neighboring pixels, this model serves as a useful comparison with the CNN.

In [ ]:
class NaiveNeuralNetwork(nn.Module):
    def __init__(self, patch_size=32):
        super().__init__()
        input_size = patch_size * patch_size * 3

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 2)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)

### Convolutional Neural Network

The CNN uses three convolutional layers with 32, 64, and 128 filters, batch normalization, max pooling, adaptive average pooling, dropout, and two fully connected layers.

**ReLU is used as the activation function; cross-entropy is the loss function.**

In [ ]:
class ConvolutionalNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))

        x = self.global_pool(x)
        x = torch.flatten(x, 1)

        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

## 7. Training and Evaluation Utilities

The original experiments used Adam with a learning rate of 0.001 and cross-entropy loss. Early stopping was based on validation loss, and the best-performing model state was retained.

The functions below consolidate the repeated training and evaluation code from the working notebook.

In [ ]:
def evaluate(model, dataloader, criterion=None):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y, _ in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            if criterion is not None:
                total_loss += criterion(logits, y).item() * y.size(0)

            predictions = logits.argmax(dim=1)
            correct += (predictions == y).sum().item()
            total += y.size(0)

    accuracy = correct / total
    average_loss = total_loss / total if criterion is not None else None

    return average_loss, accuracy

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    max_epochs=50,
    patience=3,
    learning_rate=0.001,
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    history = []

    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for x, y, _ in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * y.size(0)
            train_correct += (logits.argmax(dim=1) == y).sum().item()
            train_total += y.size(0)

        train_loss = train_loss_sum / train_total
        train_accuracy = train_correct / train_total
        val_loss, val_accuracy = evaluate(model, val_loader, criterion)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} | "
            f"train acc {train_accuracy:.2%} | "
            f"val loss {val_loss:.4f} | "
            f"val acc {val_accuracy:.2%}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print("Early stopping.")
            break

    elapsed = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"Training time: {elapsed:.2f} seconds")
    return model, history

## 8. Running an Experiment

Choose a patch size and training subset below. For the full experiments reported in the paper, the CNN was trained at 16×16, 32×32, 64×64, and 128×128. The 32×32 and 128×128 models were also retrained using 25% and 10% of the original training data.

The original full-data experiments used an early-stopping patience of 3; reduced-data experiments used a patience of 5.

In [ ]:
PATCH_SIZE = 32
TRAINING_FRACTION = 1.0  # choose 1.0, 0.25, or 0.10

if TRAINING_FRACTION == 1.0:
    experiment_paths, experiment_labels = X_train, y_train
    patience = 3
elif TRAINING_FRACTION == 0.25:
    experiment_paths, experiment_labels = X_25, y_25
    patience = 5
elif TRAINING_FRACTION == 0.10:
    experiment_paths, experiment_labels = X_10, y_10
    patience = 5
else:
    raise ValueError("TRAINING_FRACTION must be 1.0, 0.25, or 0.10.")

train_loader, val_loader, test_loader = make_loaders(
    PATCH_SIZE,
    train_paths=experiment_paths,
    train_labels=experiment_labels,
)

model = ConvolutionalNeuralNetwork().to(device)

# Training is intentionally not started automatically.
# Uncomment to reproduce an experiment:
#
# model, history = train_model(
#     model,
#     train_loader,
#     val_loader,
#     max_epochs=50,
#     patience=patience,
# )

## 9. Image-Level Majority-Vote Aggregation

Patch-level predictions are combined to obtain a single prediction for each image. An image is classified as AI-generated when at least half of its patches are predicted as AI-generated.

In [ ]:
def image_to_patches(image, patch_size):
    image = image.convert("RGB")
    transform = transforms.ToTensor()

    patches = []

    for top in range(0, image.height, patch_size):
        for left in range(0, image.width, patch_size):
            patch = image.crop(
                (left, top, left + patch_size, top + patch_size)
            )
            patches.append(transform(patch))

    return torch.stack(patches)


def predict_image(model, image, patch_size):
    patches = image_to_patches(image, patch_size).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(patches)
        patch_predictions = logits.argmax(dim=1)

    predicted_class = int(
        patch_predictions.sum().item() >= len(patch_predictions) / 2
    )

    return predicted_class, logits

In [ ]:
def aggregated_accuracy(model, image_paths, labels, patch_size):
    correct = 0

    for image_path, true_label in zip(image_paths, labels):
        image = Image.open(image_path)
        predicted_label, _ = predict_image(model, image, patch_size)

        if predicted_label == true_label:
            correct += 1

    accuracy = correct / len(image_paths)
    print(f"Image-level aggregated accuracy: {accuracy:.2%}")
    return accuracy


# Example after training/loading a patch model:
# aggregated_accuracy(model, X_test, y_test, PATCH_SIZE)

## 10. Patch-Level Interpretability

The visualization below overlays each patch according to its predicted class:

- **Green** — predicted real
- **Red** — predicted AI-generated
- Greater opacity — greater model confidence

This makes it possible to inspect which spatial regions contribute to the image-level decision.

In [ ]:
def patch_heatmap(model, image, patch_size, temperature=5.0):
    image = image.convert("RGB")
    output = image.copy()

    patches = image_to_patches(image, patch_size).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(patches)
        probabilities = F.softmax(logits / temperature, dim=1)

    predicted_label = int(
        logits.argmax(dim=1).sum().item() >= len(logits) / 2
    )

    patch_idx = 0

    for top in range(0, image.height, patch_size):
        for left in range(0, image.width, patch_size):
            box = (
                left,
                top,
                left + patch_size,
                top + patch_size,
            )

            region = output.crop(box)
            real_prob, fake_prob = probabilities[patch_idx].cpu().tolist()

            if real_prob > fake_prob:
                overlay_color = (80, 255, 120)
                confidence = real_prob
            else:
                overlay_color = (255, 80, 80)
                confidence = fake_prob

            overlay = Image.new(region.mode, region.size, overlay_color)
            region = Image.blend(region, overlay, confidence)
            output.paste(region, box)

            patch_idx += 1

    print(f"Predicted label: {predicted_label}")
    display(output)

    return predicted_label

## 11. Reported Experimental Results

The table below records the results from the original project. These values are included as reported experimental results rather than regenerated notebook output.

| Model | Test Accuracy | Aggregated Image Accuracy | Training Time |
|---|---:|---:|---:|
| 128×128 CNN | 97.10% | — | 596.13 sec |
| 64×64 Patch CNN | 94.34% | 98.01% | 676.66 sec |
| **32×32 Patch CNN** | 86.85% | **99.17%** | 3366.77 sec |
| 32×32 Fully Connected NN | 53.79% | 54.42% | 190.45 sec |
| 16×16 Patch CNN | 73.11% | 91.78% | 6852.52 sec |

The strongest image-level result came from the **32×32 patch CNN**. Although individual patches were classified at 86.85% accuracy, majority-vote aggregation increased image-level accuracy to 99.17%.

This improvement came with a substantial computational cost: the 32×32 model took considerably longer to train than the 64×64 and full-image models.

## 12. Reduced-Training-Data Results

The best-performing patch model (32×32) was compared with the full-image 128×128 CNN after reducing the training set.

| Model | Training Data | Test Accuracy | Aggregated Image Accuracy |
|---|---:|---:|---:|
| 128×128 CNN | 25% | 91.95% | — |
| 32×32 Patch CNN | 25% | 77.99% | 90.35% |
| 128×128 CNN | 10% | 90.37% | — |
| 32×32 Patch CNN | 10% | 73.55% | 86.02% |

The full-image CNN retained more accuracy as the amount of training data decreased. This suggests that the patch-based advantage observed with the full training set did not persist under the reduced-data conditions tested here.

## 13. Interpretation

A central finding is the gap between **patch-level** and **image-level** performance. Individual patches can be misclassified, but aggregating predictions across the image allows majority voting to correct many local errors.

Patch-based models also provide a useful interpretability advantage. Spatial overlays can show where the model finds evidence associated with real or synthetic imagery rather than returning only a single whole-image label.

At the same time, smaller patches discard global facial structure. The full-image model retains information such as facial symmetry and relationships among features, which may help explain its stronger performance when training data is limited.

## 14. Limitations and Future Work

Several limitations are important when interpreting these results:

- Each reported accuracy comes from a single training run rather than an average across multiple random seeds.
- Training and evaluation use images from the same underlying GAN distribution.
- Cross-generator generalization was not tested.
- Diffusion-generated faces were not included.
- Patch models may learn generator-specific local artifacts that do not transfer to newer image generators.

Future work should repeat experiments across random seeds and evaluate the models on synthetic faces produced by unseen GAN and diffusion architectures.

## Conclusion

This project demonstrates that patch-based CNN classification can combine strong image-level performance with spatial interpretability for AI-generated face detection.

On the full training set, the 32×32 patch model produced the strongest reported aggregated result, while the full-image CNN was more robust when the amount of training data was reduced. Together, these experiments illustrate the tradeoff between localized artifact detection, global image context, computational cost, and interpretability.